# SQL RAG — 쿼리를 작성·실행·수정하는 Agent

지금까지의 RAG 는 **문서(텍스트)** 를 검색했다. 이번엔 검색 대상이 **관계형 DB** 다. 자연어 질문을 받아 **SQL 을 작성 → 실행 → 에러 나면 수정** 하고, 결과를 근거로 답한다 (Text-to-SQL).

두 가지 방식으로 만든다:
1. **prebuilt ReAct 에이전트** — `SQLDatabaseToolkit` + `create_react_agent` (가장 간단)
2. **그래프 직접 구성** — 테이블 목록 → 스키마 조회 → 쿼리 생성 → 실행 → (에러 시 재생성) → 답변

```
START → chatbot ─(DB조회 필요?)─▶ list_tables → call_get_schema → get_schema
          │(불필요)→END                                              │
          ▼                                                          ▼
  answer ◀─(성공)─ check_query ◀──────────── generate_query ◀────────┘
    │                  └─(에러)─▶ generate_query (재작성)
   END
```

> `OPENAI_API_KEY` 필요. DB 는 공개 샘플(Chinook)을 다운로드한다.

## 1. 데이터베이스 연결

공개 샘플 DB **Chinook**(음반 판매점: 아티스트/앨범/트랙/고객/직원 등)을 다운로드해 SQLite 로 연결한다.

In [ ]:
import requests
import os

DB_PATH = "Chinook.db"
if not os.path.exists(DB_PATH):
    url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
    r = requests.get(url)
    if r.status_code == 200:
        with open(DB_PATH, "wb") as f:
            f.write(r.content)
        print("Chinook.db 다운로드 완료")
    else:
        print("다운로드 실패:", r.status_code)
else:
    print("Chinook.db 이미 존재")

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")
print("Dialect:", db.dialect)
print("Tables:", db.get_usable_table_names())
print("Sample:", db.run("SELECT * FROM Artist LIMIT 5;"))

## 2. SQLDatabaseToolkit — DB 조작 도구 모음

`SQLDatabaseToolkit` 은 DB 조작에 필요한 도구 4종을 묶어준다:
- `sql_db_list_tables`: 테이블 목록
- `sql_db_schema`: 특정 테이블의 스키마 + 샘플 행
- `sql_db_query`: SQL 실행 (잘못되면 에러 메시지 반환)
- `sql_db_query_checker`: 쿼리 문법 점검

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import SQLDatabaseToolkit

llm = ChatOpenAI(model="gpt-4o", temperature=0)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

for t in tools:
    print(f"- {t.name}: {t.description[:70]}")

## 3. 방식 1 — prebuilt ReAct 에이전트 (간단)

[basics 복습] `create_react_agent` 에 SQL 도구들을 주고 시스템 프롬프트로 행동 지침을 준다. 에이전트가 알아서 테이블 확인 → 스키마 조회 → 쿼리 작성 → 실행 → 답변 한다.

In [ ]:
from langgraph.prebuilt import create_react_agent

system_prompt = f"""
You are an agent designed to interact with a SQL database.
Given a question, create a syntactically correct {db.dialect} query, run it,
look at the results, and return the answer. Limit to at most 5 results unless asked.
Order by a relevant column. Never SELECT all columns, only relevant ones.
Double-check your query before executing. If you get an error, rewrite and retry.
DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP).
ALWAYS first look at the tables, then query the schema of the most relevant tables.
"""

react_agent = create_react_agent(llm, tools, prompt=system_prompt)

In [ ]:
question = "2009년에 가장 많은 매출을 올린 영업 사원은 누구인가요?"
for step in react_agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

## 4. 방식 2 — 그래프 직접 구성 (제어 명시)

ReAct 에이전트는 흐름을 LLM 에 맡긴다. 그래프로 직접 짜면 **단계를 명시적으로 제어** 할 수 있다 (테이블 목록 → 스키마 → 쿼리 생성 → 실행 → 에러 시 재생성 → 답변).

### Step 1. 도구별 ToolNode
[basics 복습] 필요한 도구를 골라 각각 `ToolNode` 로 만든다.

In [ ]:
from langgraph.prebuilt import ToolNode

list_tables_tool = next(t for t in tools if t.name == "sql_db_list_tables")
get_schema_tool = next(t for t in tools if t.name == "sql_db_schema")
run_query_tool = next(t for t in tools if t.name == "sql_db_query")

list_tables_node = ToolNode([list_tables_tool], name="list_tables")
get_schema_node = ToolNode([get_schema_tool], name="get_schema")

### Step 2. chatbot — DB 조회 시작 여부 결정
[basics 복습] `bind_tools(list_tables)` 로, DB 조회가 필요하면 tool_calls 를, 아니면 일반 답을 낸다.

In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph

def chatbot(state: MessagesState):
    llm_with_tools = llm.bind_tools([list_tables_tool])
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

### Step 3. call_get_schema — 스키마 조회 도구 호출
[basics 복습] `tool_choice="any"` 로 반드시 스키마 도구를 부르게 강제한다.

In [ ]:
def call_get_schema(state: MessagesState):
    llm_with_tools = llm.bind_tools([get_schema_tool], tool_choice="any")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

### Step 4. generate_query — 질문+스키마로 SQL 작성
에러 이력이 있으면 그걸 반영해 쿼리를 다시 만든다. 백틱 없이 SQL 문만 반환하게 한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

generate_query_system = f"""
You are an agent designed to interact with a SQL database.
Given a question, create a syntactically correct {db.dialect} query. Limit to 5 results unless asked.
Order by a relevant column. Never SELECT all columns. DO NOT make DML statements.
DO NOT wrap the response in backticks. Respond with a SQL statement only!
"""
generate_query_user = """
User input: {question}
Schema: {schema}
If an error message is given, regenerate the query based on it.
History: {history}
SQL query:"""

def generate_query(state: MessagesState):
    print("##### GENERATE QUERY #####")
    # 3번째 메시지부터(질문/스키마 이후)를 이력으로 — 이전 시도/에러 포함
    history = "\n".join(m.content for m in state["messages"][2:])
    prompt = ChatPromptTemplate.from_messages([
        ("system", generate_query_system),
        ("user", generate_query_user),
    ])
    response = llm.invoke(prompt.format_messages(
        question=state["messages"][0].content,
        schema=state["messages"][1].content,
        history=history,
    ))
    return {"messages": [response]}

### Step 5. check_query — 쿼리 실행
생성된 SQL 을 실제 실행한다. 잘못된 쿼리면 결과에 `Error:` 가 담겨 온다.

In [ ]:
def check_query(state: MessagesState):
    print("##### RUN QUERY #####")
    query = state["messages"][-1].content
    result = run_query_tool.invoke(query)
    return {"messages": [result]}

### Step 6. answer — 질문 + 쿼리 결과로 답변

In [ ]:
answer_system = """
You provide concise, accurate answers based on a context retrieved from a database via SQL.
Analyze the context and answer the user's question. ANSWER IN KOREAN.
"""

def answer(state: MessagesState):
    print("##### ANSWER #####")
    question = state["messages"][0].content
    generated_query = state["messages"][-2].content
    context = state["messages"][-1].content
    prompt = ChatPromptTemplate.from_messages([
        ("system", answer_system),
        ("user", "User Question: {question}\nSQL Query: {query}\nContext: {context}"),
    ])
    response = llm.invoke(prompt.format_messages(
        question=question, query=generated_query, context=context
    ))
    return {"messages": [response]}

### Step 7. 그래프 컴파일

[basics 복습] 실행 결과에 `Error:` 가 있으면 `generate_query` 로 되돌려 **쿼리를 수정** 하는 루프가 핵심.

In [ ]:
from langgraph.prebuilt import tools_condition

graph_builder = StateGraph(MessagesState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("list_tables", list_tables_node)
graph_builder.add_node("call_get_schema", call_get_schema)
graph_builder.add_node("get_schema", get_schema_node)
graph_builder.add_node("generate_query", generate_query)
graph_builder.add_node("check_query", check_query)
graph_builder.add_node("answer", answer)

graph_builder.add_edge(START, "chatbot")
# DB 조회 필요하면 list_tables, 아니면 종료
graph_builder.add_conditional_edges(
    "chatbot", tools_condition, {"tools": "list_tables", END: END}
)
graph_builder.add_edge("list_tables", "call_get_schema")
graph_builder.add_edge("call_get_schema", "get_schema")
graph_builder.add_edge("get_schema", "generate_query")
graph_builder.add_edge("generate_query", "check_query")

# 실행 결과에 에러가 있으면 쿼리 재생성, 없으면 답변
def should_correct(state):
    if "Error:" in state["messages"][-1].content:
        return "generate_query"
    return "answer"

graph_builder.add_conditional_edges(
    "check_query", should_correct,
    {"generate_query": "generate_query", "answer": "answer"},
)
graph_builder.add_edge("answer", END)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트

In [ ]:
question = "2009년에 가장 많은 매출을 올린 영업 사원은 누구인가요?"
for step in graph.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

## 정리

- **SQL RAG (Text-to-SQL)**: 검색 대상이 문서가 아니라 **DB**. 자연어 → SQL → 실행 → 답변
- `SQLDatabaseToolkit` 이 테이블목록/스키마/쿼리실행/문법점검 도구를 제공
- **간단**: `create_react_agent` 로 한 번에 / **명시적 제어**: 그래프로 단계 분리
- 핵심 안전장치: 실행 결과에 `Error:` 있으면 **쿼리 재생성 루프** (앞서 본 코드수정 패턴과 동일한 발상)
- DML(INSERT/UPDATE/DELETE/DROP) 금지를 프롬프트로 명시해 읽기 전용 보장

이로써 RAG 시리즈(기본 → 관련성 → 환각 → 웹보강 → SQL)를 완주했다. 문서든 DB든 "외부 지식을 근거로 답하고, 품질을 평가·교정" 하는 것이 RAG 의 핵심이다.